In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pm4py
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import holidays
from enrichment import *
from feature_engineering import *


# Data Exploration and Preparation

In [2]:
event_log = pm4py.read_xes("data/BPI Challenge 2017.xes")

/Users/samuelwidmer/Desktop/Bachelors-Thesis/.venv/lib/python3.14/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [3]:
display(event_log)
print(event_log.shape[0])

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202262,Deleted,User_1,W_Call after offers,Workflow,Workitem_1817549786,ate_abort,2017-01-06 06:33:02.212000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202263,Created,User_1,W_Call after offers,Workflow,Workitem_363876066,schedule,2017-01-06 06:33:02.221000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202264,statechange,User_28,A_Cancelled,Application,ApplState_1869071797,complete,2017-01-16 09:51:21.114000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202265,statechange,User_28,O_Cancelled,Offer,OfferState_420066181,complete,2017-01-16 09:51:21.139000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_1580299144


1202267


In [4]:
# Filter for Application events
event_log = event_log[event_log["EventOrigin"] == "Application"]
event_log.reset_index()
print(event_log.shape[0])

239595


In [5]:
display(event_log)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,2016-01-01 09:52:36.413000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,2016-01-02 11:23:04.299000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,statechange,User_52,A_Complete,Application,ApplState_946455804,complete,2016-01-02 11:30:28.633000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202247,statechange,User_1,A_Submitted,Application,ApplState_2065621861,complete,2016-12-31 21:37:54.247000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202251,statechange,User_1,A_Concept,Application,ApplState_291481081,complete,2016-12-31 21:38:27.252000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202253,statechange,User_96,A_Accepted,Application,ApplState_320370852,complete,2017-01-02 19:21:15.965000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202260,statechange,User_96,A_Complete,Application,ApplState_1120616436,complete,2017-01-02 19:27:20.474000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
event_log.info()

<class 'pandas.core.frame.DataFrame'>
Index: 239595 entries, 0 to 1202264
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   Action                 239595 non-null  object             
 1   org:resource           239595 non-null  object             
 2   concept:name           239595 non-null  object             
 3   EventOrigin            239595 non-null  object             
 4   EventID                239595 non-null  object             
 5   lifecycle:transition   239595 non-null  object             
 6   time:timestamp         239595 non-null  datetime64[ns, UTC]
 7   case:LoanGoal          239595 non-null  object             
 8   case:ApplicationType   239595 non-null  object             
 9   case:concept:name      239595 non-null  object             
 10  case:RequestedAmount   239595 non-null  float64            
 11  FirstWithdrawalAmount  0 non-null       flo

# Event Log Enrichment

In [7]:
event_log = remove_empty_columns(event_log)
event_log = add_inter_event_time(event_log)
event_log = event_add_relative_case_time(event_log)
event_log = event_add_relative_log_time(event_log)

In [8]:
display(event_log)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,time_since_prev,relative_case_time,relative_log_time
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,0.0,0,0
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,0.0,0,0
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,2016-01-01 09:52:36.413000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,0.0,0,0
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,2016-01-02 11:23:04.299000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,1.0,1,1
15,statechange,User_52,A_Complete,Application,ApplState_946455804,complete,2016-01-02 11:30:28.633000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202247,statechange,User_1,A_Submitted,Application,ApplState_2065621861,complete,2016-12-31 21:37:54.247000+00:00,Home improvement,New credit,Application_1350494635,20000.0,0.0,0,365
1202251,statechange,User_1,A_Concept,Application,ApplState_291481081,complete,2016-12-31 21:38:27.252000+00:00,Home improvement,New credit,Application_1350494635,20000.0,0.0,0,365
1202253,statechange,User_96,A_Accepted,Application,ApplState_320370852,complete,2017-01-02 19:21:15.965000+00:00,Home improvement,New credit,Application_1350494635,20000.0,1.0,1,367
1202260,statechange,User_96,A_Complete,Application,ApplState_1120616436,complete,2017-01-02 19:27:20.474000+00:00,Home improvement,New credit,Application_1350494635,20000.0,0.0,1,367


In [12]:
case_timestamps = event_log.groupby("case:concept:name")["time:timestamp"]
case_durations = case_timestamps.max() - case_timestamps.min()
avg_case_duration = case_durations.mean()
max_case_duration = case_durations.max()
min_case_duration = case_durations.min()
display(max_case_duration)
display(min_case_duration)
display(case_durations)
display(avg_case_duration)

Timedelta('169 days 01:44:19.800000')

Timedelta('0 days 00:02:11.132000')

case:concept:name
Application_1000086665   32 days 14:03:15.037000
Application_1000158214    8 days 00:47:34.434000
Application_1000311556   30 days 14:04:11.228000
Application_1000334415   13 days 15:06:16.591000
Application_1000339879   12 days 20:14:38.437000
                                   ...          
Application_999507989     6 days 07:14:35.733000
Application_999544538    79 days 18:48:35.266000
Application_999632431    31 days 18:35:34.282000
Application_999993467    13 days 18:42:42.747000
Application_999993812    17 days 16:27:25.960000
Name: time:timestamp, Length: 31509, dtype: timedelta64[ns]

Timedelta('21 days 19:57:47.486496366')

# Prefix Extraction

In [26]:



def extract_prefixes_with_time_cutoff(event_log, cutoff_in_days, target, holidays):
    all_prefixes = []
    static_attributes, dynamic_attributes = detect_static_attributes(event_log)
    reference_date = event_log["relative_log_time"].min()
    activities_to_flag = event_log["concept:name"].unique()
    for case, group in event_log.groupby("case:concept:name"):
        features = {}
        prefix = group[group["relative_case_time"] < cutoff_in_days]
        prefix_start_time = group.iloc[0]["time:timestamp"]
        prefix_end_time = group.iloc[-1]["time:timestamp"]
        if group["relative_case_time"].max() > cutoff_in_days:
            continue
        A_pending_events = group[group["concept:name"] == "A_Pending"]
        if len(A_pending_events) == 0:
            continue
        add_case_attributes(prefix, features, static_attributes)
        add_event_attributes(prefix, features, dynamic_attributes)
        add_time_features(prefix, features)
        # add_attributes_from_last_event(prefix, features)
        add_activity_counts(prefix, features, activities_to_flag)
        # add_activity_flags(prefix, features, activities_to_flag)
        add_activity_start_times(prefix, features, reference_date, activities_to_flag)
        add_time_based_features(prefix, features, prefix_start_time)
        add_prefix_stats(prefix, features, "case:concept:name")
        # add_numeric_aggregates(prefix, features)
        add_resource_features(prefix, features)
        # add_days_since_reference(prefix, features, reference_date)
        add_holiday_features(prefix, features, prefix_end_time, holidays)
        # activity_pairs = [("Send Fine", "Add penalty", "fine_to_penalty")]
        # add_pairwise_delays(prefix, features, activity_pairs)
        features["target"] = target
        all_prefixes.append(features)
    return all_prefixes

     
        
    


In [21]:
nl_holidays = holidays.NL(years=2016)
for h in nl_holidays.items():
    print(h)


(datetime.date(2016, 1, 1), 'Nieuwjaarsdag')
(datetime.date(2016, 3, 25), 'Goede Vrijdag')
(datetime.date(2016, 3, 27), 'Eerste paasdag')
(datetime.date(2016, 3, 28), 'Tweede paasdag')
(datetime.date(2016, 4, 27), 'Koningsdag')
(datetime.date(2016, 5, 5), 'Hemelvaartsdag')
(datetime.date(2016, 5, 15), 'Eerste Pinksterdag')
(datetime.date(2016, 5, 16), 'Tweede Pinksterdag')
(datetime.date(2016, 12, 25), 'Eerste Kerstdag')
(datetime.date(2016, 12, 26), 'Tweede Kerstdag')


In [29]:
import holidays
cutoff_in_days = 14
target = "A_Pending"
prefix_log = extract_prefixes_with_time_cutoff(event_log, cutoff_in_days, target, nl_holidays)

NameError: name 'detect_static_attributes' is not defined